# 01. 자료 전처리와 결과변수 정의

**연구** 임상 및 유전체 데이터 기반 뇌 위축 바이오마커 예측과 MRI 검사 대상 선별 (Healthcare AI Challenge 2026, HAI-0024)

이 노트북은 연구기획서 3절(자료, 결과변수 정의, 입력 변수)을 그대로 코드로 옮긴 것이다.

1. 임상역학(KoGES 교육용) 기반 3파일과 추적 2파일, 뇌영상 체적 3파일을 읽고 ID로 연결한다.
2. 특수코드(66666, 77777, 99999)를 변수 성격에 맞게 처리하고, 기반과 추적에 공통인 입력 변수를 같은 이름으로 정합한다.
3. 기반군을 개발군 80%와 평가군 20%로 층화 분할한다(시드 고정).
4. 개발군 전원에서 해마 비율의 연령(자연 3차 스플라인)과 성별 기대치 곡선을 적합하고, 잔차 z-점수와 위축 양성(z ≤ −1)을 산출해 평가군과 추적군에 적용한다.
5. 추적군의 두 시점 해마 체적 연간 변화율과 가속 위축(하위 20%)을 산출한다.
6. 파생 자료를 `05_분석결과/derived/`에 저장한다.

**보안 원칙** 출력 셀에는 집계값만 남긴다(개별 행 출력 금지). 원자료는 이 폴더 밖으로 내보내지 않는다.

In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
SEED = 20260915

# 프로젝트 루트: 02_Data 폴더가 있는 상위 폴더를 찾는다
ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "02_Data"
COHORT = DATA / "cohort"
BV = DATA / "뇌영상체적_합성데이터"
OUT = ROOT / "05_분석결과"
DERIVED, FIG = OUT / "derived", OUT / "figures"
DERIVED.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT.name, "| 자료 폴더 존재:", COHORT.exists(), BV.exists())

ROOT = 2609_HealthcareAIchallenge2026 | 자료 폴더 존재: True True


## 1. 임상역학 자료 읽기와 특수코드 처리

코드북 규칙: 66666 = 조사 안 함, 77777 = 해당 없음, 99999 = 미상 또는 무응답.
- 66666, 99999 → 결측.
- 77777 → 해당 변수가 "양"이나 "기간"이면 0(예: 비흡연자의 갑년, 비음주자의 음주량), 그 밖(진단 나이, 여성력, 음주 기간 범주)은 결측.

In [2]:
SPECIAL = (66666, 77777, 99999)
ZERO_IF_NA = {  # 77777(해당 없음)을 0으로 볼 변수(양, 빈도, 기간)
    "T_TAKAM","T_RICEAM","T_WINEAM","T_SOJUAM","T_BEERAM","T_HLIQAM","T_TOTALC",
    "T_SMDUYR","T_SMAM","T_PACKYR",
    "T01_TAKAM","T01_RICEAM","T01_WINEAM","T01_SOJUAM","T01_BEERAM","T01_HLIQAM","T01_SMDU","T01_SMAM",
    "T05_TAKAM","T05_RICEAM","T05_WINEAM","T05_SOJUAM","T05_BEERAM","T05_HLIQAM","T05_SMDU","T05_SMAM",
}

def read_clin(fn):
    df = pd.read_csv(COHORT / fn)
    for c in df.columns:
        if c.endswith("_ID") or c.endswith("DATA_CLASS"):
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        s = s.mask(s.isin([66666, 99999]))
        s = s.mask(s == 77777, 0.0 if c in ZERO_IF_NA else np.nan)
        df[c] = s
    return df

b1, b2, b3 = read_clin("BASE_DATA1.txt"), read_clin("BASE_DATA2.txt"), read_clin("BASE_DATA3.txt")
base = b1.merge(b2.drop(columns=["T_DATA_CLASS"]), on="T_ID").merge(b3.drop(columns=["T_DATA_CLASS"]), on="T_ID")
f01 = read_clin("FOLLOW_01_DATA.txt")
f05 = read_clin("FOLLOW_05_DATA.txt")
print("기반 임상:", base.shape, "| 추적기반:", f01.shape, "| 4차추적:", f05.shape)
print("ID 고유 여부:", base.T_ID.is_unique, f01.T00_ID.is_unique, f05.T00_ID.is_unique)

기반 임상: (10000, 61) | 추적기반: (1000, 64) | 4차추적: (698, 55)
ID 고유 여부: True True True


## 2. 입력 변수 정합

기반과 추적에서 같은 뜻의 변수를 같은 이름으로 만든다. 코드북 기준 1 = 아니오, 2 = 예 → 0/1. 흡연과 음주는 1 = 안 함, 2 = 과거, 3 = 현재 → 과거와 현재 더미. 운동은 1 = 안 한다, 2 = 한다 → 0/1. BMI는 키와 체중으로 다시 계산한다. 추적군에 없는 LDL은 Friedewald 식으로 산출한다(TG < 400).

In [3]:
def yes(s):  # 1=아니오, 2=예 -> 0/1 (결측 유지)
    return s.map({1: 0, 2: 1})

def harmonize(df, p, sex_col, age_col, base_like):
    """p: 변수 접두어('T_' 또는 'T01_'/'T05_'). base_like: 기반 코드북 변수명 사용 여부"""
    g = lambda name: df[p + name] if (p + name) in df.columns else pd.Series(np.nan, index=df.index)
    out = pd.DataFrame(index=df.index)
    out["age"] = df[age_col]
    out["male"] = (df[sex_col] == 1).astype(int)
    out["sbp"], out["dbp"] = g("SBP"), g("DBP")
    h, w = g("HEIGHT"), g("WEIGHT")
    out["bmi"] = w / (h / 100) ** 2
    out["waist"] = g("WAIST")
    smoke = g("SMOKE")
    out["smoke_past"], out["smoke_cur"] = (smoke == 2).astype(float).where(smoke.notna()), (smoke == 3).astype(float).where(smoke.notna())
    if base_like:
        out["packyr"] = g("PACKYR")
    else:  # 추적: 하루 개비 / 20 × 총 흡연기간(년)
        out["packyr"] = (g("SMAM") / 20) * g("SMDU")
    out.loc[smoke == 1, "packyr"] = 0.0
    drink = g("DRINK")
    out["drink_past"], out["drink_cur"] = (drink == 2).astype(float).where(drink.notna()), (drink == 3).astype(float).where(drink.notna())
    out["exer"] = yes(g("EXER"))
    out["htn"], out["dm"], out["lip"] = yes(g("HTN")), yes(g("DM")), yes(g("LIP"))
    if base_like:
        out["fh_htn"] = ((yes(g("FMFHT1")) == 1) | (yes(g("FMFHT2")) == 1)).astype(int)
        out["fh_dm"] = ((yes(g("FMFDM1")) == 1) | (yes(g("FMFDM2")) == 1)).astype(int)
    else:
        out["fh_htn"] = ((yes(g("FMFHT")) == 1) | (yes(g("FMMHT")) == 1)).astype(int)
        out["fh_dm"] = ((yes(g("FMFDM")) == 1) | (yes(g("FMMDM")) == 1)).astype(int)
    # 검진 혈액
    out["glu"], out["hba1c"] = g("GLU0"), g("HBA1C")
    out["cr"] = df[p + "CREATINE"] if (p + "CREATINE") in df.columns else g("CREATININE")
    out["ast"], out["alt"] = g("AST"), g("ALT")
    out["tc"], out["hdl"], out["tg"] = g("TCHL"), g("HDL"), g("TG")
    if (p + "LDL") in df.columns:
        out["ldl"] = df[p + "LDL"]
    else:
        out["ldl"] = (out["tc"] - out["hdl"] - out["tg"] / 5).where(out["tg"] < 400)
    return out

C0 = ["age", "male"]
C1 = ["sbp", "dbp", "bmi", "waist", "smoke_past", "smoke_cur", "packyr", "drink_past", "drink_cur", "exer", "htn", "dm", "lip", "fh_htn", "fh_dm"]
C2 = ["glu", "hba1c", "cr", "ast", "alt", "tc", "hdl", "ldl", "tg"]

Xb = harmonize(base, "T_", "T_SEX", "T_AGE", base_like=True)
Xb["income"], Xb["totalc"] = base["T_INCOME"], base["T_TOTALC"]   # 기반군 전용 확장 변수
Xb.insert(0, "T_ID", base["T_ID"].values)
Xf1 = harmonize(f01, "T01_", "T00_SEX", "T01_AGE", base_like=False); Xf1.insert(0, "T00_ID", f01["T00_ID"].values)
Xf1["edu"] = f01["T01_EDU"]  # 추적군 전용
Xf1["edate"] = f01["T01_EDATE"]
Xf5 = pd.DataFrame({"T00_ID": f05["T00_ID"], "age5": f05["T05_AGE"], "edate5": f05["T05_EDATE"]})

print("기반군 입력 결측률(%):")
print((Xb[C0 + C1 + C2].isna().mean() * 100).round(1).to_string())

기반군 입력 결측률(%):
age            0.0
male           0.0
sbp            0.4
dbp            0.4
bmi            0.5
waist          1.3
smoke_past     0.7
smoke_cur      0.7
packyr        12.9
drink_past     0.6
drink_cur      0.6
exer           0.6
htn            0.2
dm             0.2
lip            0.2
fh_htn         0.0
fh_dm          0.0
glu            2.4
hba1c         62.4
cr             0.0
ast            2.4
alt            2.6
tc             0.0
hdl            0.0
ldl            2.0
tg             0.3


## 3. 뇌영상 체적 읽기와 연결

비율 변수: 해마 비율 = BV_subctx_hippocampus / BV_icv, 전뇌 비율 = BV_whole_brain / BV_icv, 뇌실 비율 = BV_ventricles / BV_icv.

In [4]:
def read_bv(fn, idcol):
    df = pd.read_csv(BV / fn)
    df = df.rename(columns={df.columns[0]: idcol})
    df["hip_ratio"] = df["BV_subctx_hippocampus"] / df["BV_icv"]
    df["wb_ratio"] = df["BV_whole_brain"] / df["BV_icv"]
    df["vent_ratio"] = df["BV_ventricles"] / df["BV_icv"]
    return df

bvb = read_bv("BrainVol_BASE.csv", "T_ID")
bv1 = read_bv("BrainVol_FOLLOW_01.csv", "T00_ID")
bv5 = read_bv("BrainVol_FOLLOW_05.csv", "T00_ID")
print("뇌영상 결측 총합:", int(bvb.isna().sum().sum() + bv1.isna().sum().sum() + bv5.isna().sum().sum()))

keep = ["hip_ratio", "wb_ratio", "vent_ratio", "BV_icv", "BV_subctx_hippocampus", "BV_whole_brain", "BV_ventricles"]
Db = Xb.merge(bvb[["T_ID"] + keep], on="T_ID", how="inner")
Df1 = Xf1.merge(bv1[["T00_ID"] + keep], on="T00_ID", how="inner")
Df5 = Xf5.merge(bv5[["T00_ID"] + keep].rename(columns={k: k + "_5" for k in keep}), on="T00_ID", how="inner")
assert len(Db) == 10000 and len(Df1) == 1000 and len(Df5) == 698, "ID 연결 수가 예상과 다름"
print("연결 후 n: 기반", len(Db), "| 추적기반", len(Df1), "| 4차추적", len(Df5))
print("해마 비율 요약(기반군): 평균 %.5f, SD %.5f" % (Db.hip_ratio.mean(), Db.hip_ratio.std()))

뇌영상 결측 총합: 0
연결 후 n: 기반 10000 | 추적기반 1000 | 4차추적 698
해마 비율 요약(기반군): 평균 0.00494, SD 0.00070


## 4. 개발군과 평가군 분할

연령대(5세 구간), 성별, 대사질환 유무(고혈압, 당뇨, 고지혈증 중 하나 이상)를 결합한 층으로 80:20 층화 분할한다. 시드는 20260915로 고정하고 분할 결과를 저장한다.

In [5]:
Db["any_dz"] = ((Db.htn == 1) | (Db.dm == 1) | (Db.lip == 1)).astype(int)
Db["age_band"] = (Db.age // 5 * 5).clip(upper=75)
strata = Db.age_band.astype(str) + "_" + Db.male.astype(str) + "_" + Db.any_dz.astype(str)
rare = strata.value_counts()[lambda s: s < 10].index
strata = strata.where(~strata.isin(rare), "rare")   # 10명 미만 층은 하나로 합침
dev_idx, eval_idx = train_test_split(Db.index, test_size=0.20, random_state=SEED, stratify=strata)
Db["split"] = "dev"; Db.loc[eval_idx, "split"] = "eval"
print(Db.split.value_counts().to_dict())
print("층별 균형 점검 — 평가군 비율(연령대×성별×질환):")
print(Db.groupby(["age_band", "male", "any_dz"]).split.apply(lambda s: round((s == "eval").mean(), 2)).unstack([1, 2]).to_string())

{'dev': 8000, 'eval': 2000}
층별 균형 점검 — 평가군 비율(연령대×성별×질환):
male        0           1      
any_dz      0     1     0     1
age_band                       
40        0.2  0.20  0.20  0.20
45        0.2  0.20  0.20  0.21
50        0.2  0.20  0.20  0.20
55        0.2  0.20  0.20  0.20
60        0.2  0.20  0.20  0.20
65        0.2  0.20  0.20  0.20
70        0.2  0.19  0.20  0.20
75        0.2  0.19  0.21  0.17


## 5. 단면 결과변수: 연령 및 성별 기대치 곡선과 z-점수

개발군 전원에서 `비율 ~ cr(age, df=4) + male`(자연 3차 스플라인)을 적합한다. 잔차 SD는 개발군 단일값. z ≤ −1을 위축 양성으로 한다(뇌실은 z ≥ +1).
사전 규칙 점검: (a) 성별×연령 상호작용 F-검정 (b) 연령대별 잔차 SD.

In [6]:
dev = Db[Db.split == "dev"]

def fit_norm(target):
    m = smf.ols(f"{target} ~ cr(age, df=4) + male", data=dev).fit()
    m_int = smf.ols(f"{target} ~ cr(age, df=4) * male", data=dev).fit()
    f_test = m_int.compare_f_test(m)  # (F, p, df_diff)
    sd = float(np.sqrt(m.scale))
    return m, sd, f_test

def apply_norm(m, sd, df, target, age_col="age"):
    tmp = df[[age_col, "male"]].rename(columns={age_col: "age"})
    return (df[target].values - m.predict(tmp).values) / sd

norm_models, norm_info = {}, {}
for tgt in ["hip_ratio", "wb_ratio", "vent_ratio"]:
    m, sd, (F, p, dfd) = fit_norm(tgt)
    norm_models[tgt] = m
    # 연령대별 잔차 SD (개발군)
    res = dev[tgt] - m.fittedvalues
    sd_by_age = res.groupby(dev.age_band).std().round(6).to_dict()
    norm_info[tgt] = {"resid_sd": sd, "R2": round(m.rsquared, 4), "sex_x_age_F": round(F, 3), "sex_x_age_p": round(p, 4), "sd_by_age_band": sd_by_age,
                      "coef_male": round(float(m.params["male"]), 6)}
    print(f"[{tgt}] R²={m.rsquared:.3f}  잔차SD={sd:.6f}  성별계수={m.params['male']:.6f}  성별×연령 F={F:.2f} p={p:.4f}")
    print("   연령대별 잔차 SD:", {k: round(v / sd, 2) for k, v in sd_by_age.items()}, "(단일 SD 대비 배수)")

for tgt, short in [("hip_ratio", "hip"), ("wb_ratio", "wb"), ("vent_ratio", "vent")]:
    m, sd = norm_models[tgt], norm_info[tgt]["resid_sd"]
    Db[f"z_{short}"] = apply_norm(m, sd, Db, tgt)
    Df1[f"z_{short}"] = apply_norm(m, sd, Df1, tgt)
    Df5tmp = Df5.rename(columns={f"{tgt}_5": tgt, "age5": "age"}).merge(Df1[["T00_ID", "male"]], on="T00_ID")
    Df5[f"z_{short}_5"] = apply_norm(m, sd, Df5tmp, tgt)

Db["atrophy_pos"] = (Db.z_hip <= -1).astype(int)
Db["atrophy_pos_b20"] = (Db.z_hip <= Db.loc[Db.split == "dev", "z_hip"].quantile(0.20)).astype(int)
Db["wb_pos"] = (Db.z_wb <= -1).astype(int); Db["vent_pos"] = (Db.z_vent >= 1).astype(int)
Db["composite_pos"] = ((Db.atrophy_pos == 1) | (Db.wb_pos == 1)).astype(int)
Df1["atrophy_pos"] = (Df1.z_hip <= -1).astype(int)

print("\n위축 양성 유병률: 개발군 %.1f%%, 평가군 %.1f%%, 추적기반 %.1f%%" % (
    100 * Db.loc[Db.split == "dev", "atrophy_pos"].mean(), 100 * Db.loc[Db.split == "eval", "atrophy_pos"].mean(), 100 * Df1.atrophy_pos.mean()))
print("z_hip 평균/SD: 개발군 %.3f/%.3f, 평가군 %.3f/%.3f, 추적기반 %.3f/%.3f" % (
    Db.loc[Db.split == "dev", "z_hip"].mean(), Db.loc[Db.split == "dev", "z_hip"].std(),
    Db.loc[Db.split == "eval", "z_hip"].mean(), Db.loc[Db.split == "eval", "z_hip"].std(), Df1.z_hip.mean(), Df1.z_hip.std()))
d = Db[Db.split == "dev"]
print("보조 결과 유병률(개발군): 전뇌 %.1f%%, 뇌실 %.1f%%, 복합 %.1f%%" % (100 * d.wb_pos.mean(), 100 * d.vent_pos.mean(), 100 * d.composite_pos.mean()))

[hip_ratio] R²=0.313  잔차SD=0.000587  성별계수=-0.000323  성별×연령 F=6.71 p=0.0000
   연령대별 잔차 SD: {40: 1.03, 45: 1.02, 50: 1.0, 55: 1.02, 60: 0.97, 65: 0.95, 70: 0.95, 75: 1.02} (단일 SD 대비 배수)
[wb_ratio] R²=0.265  잔차SD=0.053847  성별계수=-0.000789  성별×연령 F=1.91 p=0.1058
   연령대별 잔차 SD: {40: 1.02, 45: 1.02, 50: 1.01, 55: 0.99, 60: 0.96, 65: 0.98, 70: 0.99, 75: 0.9} (단일 SD 대비 배수)
[vent_ratio] R²=0.525  잔차SD=0.004572  성별계수=-0.000060  성별×연령 F=42.98 p=0.0000
   연령대별 잔차 SD: {40: 0.96, 45: 1.1, 50: 1.16, 55: 1.05, 60: 0.85, 65: 0.68, 70: 0.64, 75: 0.55} (단일 SD 대비 배수)

위축 양성 유병률: 개발군 15.2%, 평가군 15.3%, 추적기반 14.6%
z_hip 평균/SD: 개발군 0.006/1.000, 평가군 0.005/0.982, 추적기반 -0.015/0.965
보조 결과 유병률(개발군): 전뇌 15.4%, 뇌실 15.6%, 복합 24.6%


## 6. 종단 결과변수: 두 시점 연간 변화율

대칭 백분율 변화 = (V5 − V1) / ((V5 + V1)/2) × 100, 경과연수 = (T05_EDATE − T01_EDATE)의 개월 수 / 12, 연간 변화율 = 변화 / 경과연수. 추적군 698명 안에서 하위 20%를 가속 위축으로 한다.

In [7]:
def ym_to_months(v):
    v = int(v); return (v // 100) * 12 + (v % 100)

L = Df1.merge(Df5, on="T00_ID", how="inner")
L["years"] = (L.edate5.map(ym_to_months) - L.edate.map(ym_to_months)) / 12
for v, short in [("BV_subctx_hippocampus", "hip"), ("BV_whole_brain", "wb"), ("BV_ventricles", "vent")]:
    v1, v5 = L[v], L[v + "_5"]
    L[f"spc_{short}"] = (v5 - v1) / ((v5 + v1) / 2) * 100
    L[f"rate_{short}"] = L[f"spc_{short}"] / L.years
L["accel_atrophy"] = (L.rate_hip <= L.rate_hip.quantile(0.20)).astype(int)
L["icv_change_pct"] = (L.BV_icv_5 - L.BV_icv) / L.BV_icv * 100

print("추적군 n =", len(L))
print("경과연수: 평균 %.2f, 최소 %.2f, 최대 %.2f" % (L.years.mean(), L.years.min(), L.years.max()))
print("해마 연간 변화율(%%/년): 평균 %.3f, SD %.3f, 하위 20%% 경계 %.3f" % (L.rate_hip.mean(), L.rate_hip.std(), L.rate_hip.quantile(0.20)))
print("전뇌 연간 변화율(%%/년): 평균 %.3f, SD %.3f | 뇌실: 평균 %.3f, SD %.3f" % (L.rate_wb.mean(), L.rate_wb.std(), L.rate_vent.mean(), L.rate_vent.std()))
print("ICV 시점 간 변화(%%): 평균 %.3f, SD %.3f  (0에 가까워야 정상)" % (L.icv_change_pct.mean(), L.icv_change_pct.std()))
print("단면 위축 양성(추적기반) 여부별 해마 연간 변화율 평균: 양성 %.3f, 음성 %.3f" % (
    L.loc[L.atrophy_pos == 1, "rate_hip"].mean(), L.loc[L.atrophy_pos == 0, "rate_hip"].mean()))
print("가속 위축군에서 단면 위축 양성 비율: %.1f%% (전체 %.1f%%)" % (100 * L.loc[L.accel_atrophy == 1, "atrophy_pos"].mean(), 100 * L.atrophy_pos.mean()))

추적군 n = 698
경과연수: 평균 10.76, 최소 9.00, 최대 11.83
해마 연간 변화율(%/년): 평균 -0.670, SD 0.537, 하위 20% 경계 -1.089
전뇌 연간 변화율(%/년): 평균 -0.404, SD 0.355 | 뇌실: 평균 3.285, SD 3.099
ICV 시점 간 변화(%): 평균 0.140, SD 3.918  (0에 가까워야 정상)
단면 위축 양성(추적기반) 여부별 해마 연간 변화율 평균: 양성 -0.405, 음성 -0.709
가속 위축군에서 단면 위축 양성 비율: 9.3% (전체 12.9%)


## 7. 저장

- `base_analysis.csv`: 기반군 10,000명 — ID, 분할, 입력 변수(C0, C1, C2, 확장), 비율, z-점수, 결과 플래그
- `follow_baseline.csv`: 추적기반 1,000명 — 입력 변수, 비율, z-점수, 위축 양성
- `follow_longitudinal.csv`: 698명 — 두 시점 체적, 경과연수, 변화율, 가속 위축
- `normative_model.json`: 기대치 곡선 정보(잔차 SD, R², 상호작용 검정, 연령대별 SD)와 변수 블록 정의

In [8]:
Db.to_csv(DERIVED / "base_analysis.csv", index=False)
Df1.to_csv(DERIVED / "follow_baseline.csv", index=False)
L.to_csv(DERIVED / "follow_longitudinal.csv", index=False)
import pickle
with open(DERIVED / "normative_models.pkl", "wb") as f:
    pickle.dump({k: v for k, v in norm_models.items()}, f)
with open(DERIVED / "normative_model.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "blocks": {"C0": C0, "C1": C1, "C2": C2, "base_only": ["income", "totalc"], "follow_only": ["edu"]},
               "outcome": {"primary": "atrophy_pos = z_hip <= -1", "sensitivity": "atrophy_pos_b20 (개발군 하위 20%)",
                           "secondary": ["wb_pos = z_wb <= -1", "vent_pos = z_vent >= 1", "composite_pos"],
                           "longitudinal": "accel_atrophy = rate_hip 하위 20% (추적군 내)"},
               "normative": norm_info}, f, ensure_ascii=False, indent=2, default=str)
print("저장 완료:", sorted(p.name for p in DERIVED.iterdir()))

저장 완료: ['base_analysis.csv', 'follow_baseline.csv', 'follow_longitudinal.csv', 'normative_model.json', 'normative_models.pkl']
